# Module Coherence — Audit du Data Warehouse

Ce notebook analyse et documente le script `check_coherence.py`.

**Rôle :** S'assurer que le chargement des données dans la base de données (modélisation dimensionnelle) s'est effectué sans perte ni duplication.
**Principes :**
*   **Complétude temporelle :** Identifier les "trous" de données (heures manquantes).
*   **Validation de la modélisation :** Vérifier la règle stricte du projet : `nombre de lignes de la table de faits ≈ nombre de villes × nombre d'heures couvertes`.

In [14]:
# 0. Configuration Colab
!pip install psycopg2-binary -q

import sys
import pandas as pd
import psycopg2
from datetime import datetime, timezone

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Mode "Documentation" dans Colab : on ne tente aucune connexion.
    # On force l'URL à None pour désactiver l'exécution SQL.
    DATABASE_URL = None
    REPORT_PATH = "coherence_report.md"
else:
    # En local (sur le serveur de production), on charge la vraie config
    from config import DATABASE_URL
    REPORT_PATH = "data/coherence_report.md"

print("Configuration terminée ✓")

Configuration terminée ✓


### 1. Fonctions de requêtage SQL

Ces fonctions internes interrogent directement les tables de faits et de dimensions pour extraire les volumétries et les bornes temporelles.

In [15]:
def _connect():
    return psycopg2.connect(DATABASE_URL)

def _bounds(conn):
    """Bornes temporelles globales couvertes par le warehouse (toutes villes confondues)."""
    with conn.cursor() as cur:
        cur.execute("SELECT MIN(timestamp_utc), MAX(timestamp_utc) FROM dim_temps")
        min_ts, max_ts = cur.fetchone()
    return min_ts, max_ts

def _counts_par_ville(conn):
    """Ramène le nombre de lignes réelles par ville depuis la table de faits."""
    query = """
        SELECT v.nom, v.pays, COUNT(f.*) AS n_lignes,
               MIN(t.timestamp_utc) AS premiere_mesure,
               MAX(t.timestamp_utc) AS derniere_mesure
        FROM dim_ville v
        LEFT JOIN fact_qualite_air f ON f.id_ville = v.id_ville
        LEFT JOIN dim_temps t ON t.id_temps = f.id_temps
        GROUP BY v.nom, v.pays
        ORDER BY v.nom
    """
    return pd.read_sql(query, conn)

def _heures_manquantes(conn, nom_ville: str, min_ts, max_ts) -> int:
    """Compte les heures théoriques où la ville n'a AUCUNE ligne de faits (trous réels)."""
    query = """
        WITH heures_theoriques AS (
            SELECT generate_series(
                date_trunc('hour', %(min_ts)s::timestamptz),
                date_trunc('hour', %(max_ts)s::timestamptz),
                interval '1 hour'
            ) AS heure
        ),
        heures_presentes AS (
            SELECT date_trunc('hour', t.timestamp_utc) AS heure
            FROM fact_qualite_air f
            JOIN dim_temps t ON t.id_temps = f.id_temps
            JOIN dim_ville v ON v.id_ville = f.id_ville
            WHERE v.nom = %(nom)s
        )
        SELECT COUNT(*) FROM heures_theoriques ht
        LEFT JOIN heures_presentes hp ON hp.heure = ht.heure
        WHERE hp.heure IS NULL
    """
    with conn.cursor() as cur:
        cur.execute(query, {"min_ts": min_ts, "max_ts": max_ts, "nom": nom_ville})
        (n,) = cur.fetchone()
    return n

print("Fonctions SQL définies ✓")

Fonctions SQL définies ✓


### 2. Logique d'audit et génération du rapport
`check_coherence()` consolide les données, calcule l'écart global et génère le fichier markdown final.

In [16]:
def check_coherence(write_report: bool = True):
    # --- LE NOUVEAU BOUCLIER DE SÉCURITÉ EST ICI ---
    if not DATABASE_URL:
        print("❌ Audit annulé : Mode 'Documentation'. L'URL de la base de données est absente pour protéger le projet.")
        return pd.DataFrame()
    # -----------------------------------------------

    conn = _connect()
    try:
        min_ts, max_ts = _bounds(conn)
        if min_ts is None:
            print("[coherence] dim_temps est vide — rien à vérifier.")
            return pd.DataFrame()

        heures_theoriques_totales = int((max_ts - min_ts).total_seconds() // 3600) + 1

        df = _counts_par_ville(conn)
        df["heures_theoriques"] = heures_theoriques_totales
        df["heures_manquantes"] = df["nom"].apply(
            lambda nom: _heures_manquantes(conn, nom, min_ts, max_ts)
        )
        df["taux_couverture_pct"] = (
            (1 - df["heures_manquantes"] / df["heures_theoriques"]) * 100
        ).round(2)

        print(f"Période couverte (globale) : {min_ts} -> {max_ts}")
        print(f"Heures théoriques par ville : {heures_theoriques_totales}\n")

        display(df[["nom", "pays", "n_lignes", "heures_manquantes", "taux_couverture_pct"]])

        total_attendu = heures_theoriques_totales * len(df)
        total_reel = int(df["n_lignes"].sum())

        print(f"\nTotal attendu (villes x heures) : {total_attendu}")
        print(f"Total réel (lignes de faits)    : {total_reel}")

        ecart = total_attendu - total_reel
        pct_ecart = 100 * ecart / total_attendu if total_attendu > 0 else 0
        print(f"Écart global                    : {ecart} ({pct_ecart:.2f} %)")

        if write_report:
            _write_markdown(df, min_ts, max_ts, heures_theoriques_totales, total_attendu, total_reel)
            print(f"\n[coherence] Rapport écrit dans {REPORT_PATH}")

        return df
    finally:
        conn.close()

### 3. Exécution de l'Audit

In [17]:
resultats = check_coherence(write_report=True)

❌ Audit annulé : Mode 'Documentation'. L'URL de la base de données est absente pour protéger le projet.
